# Suite VLGDEF — Verilog definitions

`Definitions.define(unit)` gives the entities a source text declares, in its scopes, as IEEE 1800 resolves
names; `referents` tells what a name refers to.

In [ ]:
import { readFileSync } from "node:fs";

import { Syntax as F } from "@mbse/programs/Framework";
import { Definitions as D, SystemVerilog2023, Syntax as S } from "@mbse/programs/Verilog";

import { assert, equal } from "./support.js";

const SOURCES = "../../conformance/sources";
function define(text: string): [S.SourceText, ReturnType<typeof D.define>] {
  const unit = SystemVerilog2023.parse(text);
  return [unit, D.define(unit)];
}
const names = (unit: S.SourceText, spelling: string) => [...F.walk(unit)].filter((n) => n instanceof S.NameExpression
  && n.name instanceof S.Identifier && n.name.spelling === spelling);
const show = (entities: Iterable<unknown>) => [...entities].map(String);
let unit: S.SourceText, program: ReturnType<typeof D.define>, entities: string[];

## VLGDEF-01 · Every entity of the conformance source, with its kind and qualified name

In [ ]:
unit = SystemVerilog2023.parse(readFileSync(`${SOURCES}/systemverilog2023.sv`, "utf8"), [SOURCES]);
program = D.define(unit);
entities = show(program.entities());
equal(entities.slice(0, 6), ["<package logger_pkg>", "<parameter logger_pkg::FIELDS>", "<localparam logger_pkg::BATTERY_LOW>",
  "<enumerator logger_pkg::IDLE>", "<enumerator logger_pkg::SAMPLE>", "<enumerator logger_pkg::SEND>"]);
assert(entities.includes("<argument logger_pkg::in_range::raw>") && entities.includes("<block sampler.counter>"));
assert(entities.includes("<localparam sampler.four.HALF>") && entities.includes("<instance sampler.u_fifo>"));
const unresolved = [...new Set([...F.walk(unit)].filter((n) => n instanceof S.NameExpression && n.name instanceof S.Identifier
  && D.referents(program, n).length === 0).map((n: any) => n.name.spelling))].sort();
equal(unresolved, ["tb"]);
console.log(entities.length, "entities");

## VLGDEF-02 · Imports: explicit ones alias, wildcard ones are found where nothing nearer is

In [ ]:
[unit, program] = define(`package p; parameter A = 1; parameter B = 2; parameter C = 3; endpackage
package q; parameter C = 4; endpackage
module m; import p::A; import q::*; import p::missing; import nowhere::*; import nowhere::Y;
parameter B = 5;
initial x = A + B + C + p::B + nowhere::X + p::B2;
endmodule`);
equal(show(D.referents(program, names(unit, "A")[0])), ["<parameter p::A>"]);
equal(show(D.referents(program, names(unit, "B")[0])), ["<parameter m.B>"]);
equal(show(D.referents(program, names(unit, "C")[0])), ["<parameter q::C>"]);
const scoped = [...F.walk(unit)].filter((n) => n instanceof S.ScopedName);
equal(show(D.referents(program, scoped[0])), ["<parameter p::B>"]);
assert(D.referents(program, scoped[1]).length === 0 && D.referents(program, scoped[2]).length === 0);
const missing = [...program.entities()].filter((e) => e.name === "missing")[0];
assert(missing !== undefined && missing.kind === "import" && missing.target === null);
console.log("ok");

## VLGDEF-03 · Ports, parameters, types, enumerators, functions and blocks, each in its scope

In [ ]:
[unit, program] = define(`module m #(parameter type T = int, parameter N) (a, b);
input a; output b; typedef enum {X, Y} e_t;
genvar g;
function int f(int v); begin : inner int t; t = v; end return v; endfunction
initial begin int w; for (int i = 0; i < 2; i++) w = i; foreach (q[j]) w = j; for (;;) ; end
for (g = 0; g < 2; g++) begin : gen end
endmodule`);
entities = show(program.entities());
equal(entities.slice(0, 7), ["<module m>", "<type parameter m.T>", "<parameter m.N>", "<port m.a>", "<port m.b>",
  "<enumerator m.X>", "<enumerator m.Y>"]);
const port = [...program.entities()].filter((e) => e.name === "a")[0];
assert(port !== undefined && port.declarations.length === 2 && port.definition instanceof S.PortReference);
for (const entity of ["<function m.f>", "<argument m.f.v>", "<block m.f.inner>", "<variable m.f.inner.t>", "<block m.gen>",
  "<genvar m.g>"]) {
  assert(entities.includes(entity), entity);
}
equal(D.referents(program, names(unit, "i")[0]).map((e) => e.kind), ["variable"]);
equal(D.referents(program, names(unit, "j")[0]).map((e) => e.kind), ["variable"]);
equal(D.referents(program, names(unit, "q")[0]), []);
const declared = [...F.walk(unit)].filter((n) => n instanceof S.Identifier && n.spelling === "w")[0];
equal(show(D.referents(program, declared)), ["<variable w>"]);
equal(D.referents(program, new S.Identifier({ spelling: "z" })), []);
console.log("ok");

## VLGDEF-04 · Interfaces, modports, instances and out-of-block names

In [ ]:
[, program] = define(`interface bus; logic d; modport mp (input d); endinterface
module top; bus b(); sub u (.p(b.d)); endmodule
function int c::f(); endfunction`);
entities = show(program.entities());
assert(entities.includes("<modport bus.mp>") && entities.includes("<instance top.b>") && entities.includes("<instance top.u>"));
assert(![...program.entities()].some((e) => e.kind === "function" && e.name === "f"));
console.log("ok");